<a href="https://colab.research.google.com/github/NVHau-K14/Tuan03_ThucHanh_DeepLearning/blob/main/ANN_DuBaoDanhGiaChatLuongXeOto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# ==============================================================================
# 1. TẢI VÀ ĐỌC DỮ LIỆU TỪ UCI MACHINE LEARNING REPOSITORY
# ==============================================================================
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv(url, names=columns)

print("--- 5 dòng dữ liệu đầu tiên ---")
print(df.head())
print("\nPhân bố các nhãn mục tiêu:")
print(df['class'].value_counts())

# ==============================================================================
# 2. TIỀN XỬ LÝ DỮ LIỆU (PREPROCESSING)
# ==============================================================================
# Tách thuộc tính (X) và nhãn mục tiêu (y)
X = df.drop(columns=['class'])
y = df['class']
encoder_X = OrdinalEncoder()
X_encoded = encoder_X.fit_transform(X)

# Nhãn mục tiêu 'class' có 4 phân lớp (unacc, acc, good, vgood)
encoder_y = OneHotEncoder(sparse_output=False)
y_encoded = encoder_y.fit_transform(y.values.reshape(-1, 1))

# Chia tập dữ liệu thành 80% Huấn luyện (Train) và 20% Kiểm thử (Test)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_split=0.2, random_state=42)

# ==============================================================================
# 3. XÂY DỰNG MÔ HÌNH ANN
# ==============================================================================
model = tf.keras.models.Sequential([
    # Input layer nhận vào 6 thuộc tính tương ứng với 6 cột
    tf.keras.layers.Input(shape=(6,)),

    # Hidden layer 1: 32 nút, hàm kích hoạt ReLU
    tf.keras.layers.Dense(32, activation='relu'),

    # Hidden layer 2: 16 nút, hàm kích hoạt ReLU
    tf.keras.layers.Dense(16, activation='relu'),

    # Output layer: 4 nút (tương ứng với 4 lớp chất lượng xe), hàm kích hoạt Softmax
    tf.keras.layers.Dense(4, activation='softmax')
])

# Biên dịch mô hình
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==============================================================================
# 4. HUẤN LUYỆN MÔ HÌNH (TRAINING)
# ==============================================================================
print("\n--- Bắt đầu huấn luyện mô hình ---")
history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=16,
    validation_split=0.1, # Trích 10% từ tập train làm tập validation để theo dõi overfitting
    verbose=1
)

# ==============================================================================
# 5. ĐÁNH GIÁ MÔ HÌNH (EVALUATION)
# ==============================================================================
print("\n--- Đánh giá trên tập kiểm thử (Test Set) ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Độ chính xác (Accuracy): {accuracy*100:.2f}%")
print(f"Giá trị mất mát (Loss): {loss:.4f}")

# Dự đoán kết quả trên tập Test
y_pred = model.predict(X_test)
# Chuyển từ xác suất (Softmax) sang chỉ số lớp có xác suất cao nhất (Argmax)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Lấy danh sách tên các nhãn để hiển thị báo cáo
class_names = encoder_y.categories_[0]

print("\n--- Báo cáo chi tiết (Classification Report) ---")
print(classification_report(y_test_classes, y_pred_classes, target_names=class_names))

# ==============================================================================
# 6. TRỰC QUAN HÓA QUÁ TRÌNH HỌC (PLOTTING)
# ==============================================================================
plt.figure(figsize=(12, 4))

# Đồ thị độ chính xác (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Độ chính xác qua các Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Đồ thị mất mát (Loss)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Giá trị Loss qua các Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()